In [ ]:
import os
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embedding_model = OpenAIEmbeddings(
    model=os.environ['EMBEDDING_MODEL'],
    base_url=os.environ['EMBEDDING_BASE_URL'],
    api_key=lambda :os.environ['EMBEDDING_API_KEY'],
    check_embedding_ctx_length=False,
)

embedding_model.embed_query("你好")

c:\Users\zengd\Desktop\lc_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[0.014448204077780247,
 -0.010502799414098263,
 0.004729184787720442,
 -0.03969452157616615,
 -0.004232778213918209,
 -0.015298440121114254,
 -0.01623697392642498,
 0.01807224005460739,
 -0.03826238960027695,
 0.03904728591442108,
 -0.022602422162890434,
 -0.020131412893533707,
 0.04615578055381775,
 -0.023457108065485954,
 -0.006351256277412176,
 0.009438722394406796,
 -0.0408836267888546,
 -0.015478970482945442,
 0.02635236829519272,
 0.011220032349228859,
 0.00805592816323042,
 -0.012824410572648048,
 0.0040717776864767075,
 -0.010296708904206753,
 0.024779677391052246,
 0.02075432427227497,
 -0.02428109385073185,
 -0.015797989442944527,
 -0.009116534143686295,
 0.015315803699195385,
 -0.03913159668445587,
 -0.005564128048717976,
 0.01180995348840952,
 0.027562826871871948,
 -0.007294442038983107,
 0.02986888587474823,
 -0.008149824105203152,
 -0.013018177822232246,
 0.016805121675133705,
 -0.02220628410577774,
 0.006134764291346073,
 -0.01162002980709076,
 6.381572893587872e-05,
 -

: 

In [2]:
from langchain_core.documents import Document

# 1. 准备示例文档数据
docs = [
    Document(page_content="iPhone 15 Pro 采用了钛金属边框，搭载 A17 Pro 芯片。"),
    Document(page_content="苹果公司的最新智能手机电池续航得到了大幅提升。"),
    Document(page_content="华为 Mate 60 Pro 支持卫星通话功能，采用麒麟芯片。"),
    Document(page_content="特斯拉 Model 3 是一部纯电动轿车，续航里程较长。"),
]

In [3]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(docs, embedding_model)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [4]:
vector_retriever.invoke("卫星通话") 

[Document(id='7aad40b7-280f-4253-9043-a68629c31217', metadata={}, page_content='华为 Mate 60 Pro 支持卫星通话功能，采用麒麟芯片。'),
 Document(id='dbb08674-0b19-4cd8-88ba-c87f4e3f4e7a', metadata={}, page_content='苹果公司的最新智能手机电池续航得到了大幅提升。'),
 Document(id='240991ee-0c06-4146-b06e-3cce1b34177a', metadata={}, page_content='iPhone 15 Pro 采用了钛金属边框，搭载 A17 Pro 芯片。'),
 Document(id='a454288b-2a13-49cf-9b20-7695e025e5ad', metadata={}, page_content='特斯拉 Model 3 是一部纯电动轿车，续航里程较长。')]

In [6]:
import jieba
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers import EnsembleRetriever

# 2. 构建 BM25 检索器（稀疏检索）
# 注意：若针对中文，建议传入分词函数 preprocess_func=jieba.lcut
bm25_retriever = BM25Retriever.from_documents(docs, preprocess_func=jieba.lcut)
bm25_retriever.k = 5  # 单独召回前 5 条

# 4. 使用 EnsembleRetriever 进行 RRF 融合
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],  # 设置融合权重（默认为平等 RRF 融合）
)

# 5. 执行混合检索
query = "苹果手机的芯片和电池怎么样？"
results = ensemble_retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content}")

[1] 苹果公司的最新智能手机电池续航得到了大幅提升。
[2] 特斯拉 Model 3 是一部纯电动轿车，续航里程较长。
[3] iPhone 15 Pro 采用了钛金属边框，搭载 A17 Pro 芯片。
[4] 华为 Mate 60 Pro 支持卫星通话功能，采用麒麟芯片。
